In [1]:
from neuralhydrology.datautils.burn_areas import compute_burn_area
from pathlib import Path
from neuralhydrology.utils.eeutils import initialize_earth
import ee
from neuralhydrology.utils.config import Config
from neuralhydrology.modelzoo.revnetmodified import ModifiedRevnet
import visualtorch
from neuralhydrology.datasetzoo.camelsusburn import CamelsUSBurn
import pandas as pd
import numpy as np




In [2]:
initialize_earth(Path("../../credentials/ee-credentials.json"))

In [3]:
# Filter collection for the target event
fire_fc = ee.FeatureCollection("JRC/GWIS/GlobFire/v2/FinalPerimeters")
# fire_fc = fire_fc.filter(ee.Filter.eq("Id", "FL2754808157719850109"))

ss = fire_fc.first().getInfo()
print(ss)

# Option A: Format directly on the GEE server side
#end_date_str = ee.Date(fire_fc.first().get("FinalDate")).format("YYYY-MM-dd").getInfo()

#print("Fire End Date:", end_date_str)

{'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-36.002467081870314, -9.341666699687908], [-36.002897856375554, -9.345833313876454], [-35.998675200709414, -9.345833309898394], [-35.998244402953304, -9.341666649858388], [-36.002467081870314, -9.341666699687908]]]}, 'id': '0000000000000000c9ba', 'properties': {'FDate': 1197763198976, 'IDate': 1197763198976, 'Id': 9331845}}


### Checking the revnet modified model


In [3]:
cfg = Config(Path("1_basin.yml"))
    # Add additional configuration
dict_config = cfg.as_dict()
dict_config.update({
        "daily.input_size":7, 
        "daily.hidden_size":20, 
        "burn.input_size":5,
        "burn.hidden_size1":4, 
        "burn.hidden_size2":3,
        "burn.kernel_size1":5, 
        "burn.kernel_size2":5, 
        "burn_area_resolution_days":30, 
        "train_basin_file": "1_basin.txt",
        "data_dir":"../../data/CAMELS_US",
        "hidden_size":20, 
        "burn_start_dates":{"01013500":"1995-12"},
        "burn_end_dates":{"01013500":"2010-01"},
        "month_emb_dim":4
        
        })
cfg = Config(dict_config, allow_unknown_keys=True) # Update config
    


### Load the dataset with burn data

In [4]:
dataset = CamelsUSBurn(cfg, is_train=True, period="train" )

INFO:Camels US:Loading burn timeseries data
INFO:neuralhydrology.datasetzoo.basedataset:Loading basin data into xarray data set.
100%|██████████| 119/119 [00:17<00:00,  6.97it/s]
INFO:Camels US:Finished loading flow and forcings time series data. Preparing to load burn timeseries data
INFO:Camels US:Finished loading burn time series data
INFO:Camels US:Creating lookup tables for forcings and flow data
INFO:neuralhydrology.datasetzoo.basedataset:Create lookup table and convert to pytorch tensor
100%|██████████| 119/119 [00:05<00:00, 23.61it/s]
INFO:Camels US:Finished creating lookup tables for forcings and flow data
INFO:Camels US:Enriching lookup tables with burn data
100%|██████████| 119/119 [00:01<00:00, 91.95it/s]
INFO:Camels US:Adding indexes for burn data corresponding to the end days in forcing date for each sample
100%|██████████| 391272/391272 [00:01<00:00, 283936.00it/s]
INFO:Camels US:Finished enriching lookup tables with burn data


In [5]:
dataloaded = [dataset[i] for i in np.random.choice(len(dataset), size=20)]
collated = dataset.collate_fn(dataloaded)    

# Make one run of the data

In [6]:
revnet = ModifiedRevnet(cfg)
output = revnet(collated)
output

{'y_hat': tensor([[[ 0.1259],
          [ 0.1073],
          [ 0.0587],
          ...,
          [-0.1196],
          [-0.1378],
          [-0.1468]],
 
         [[ 0.1774],
          [ 0.0659],
          [ 0.0748],
          ...,
          [ 0.1459],
          [ 0.1186],
          [ 0.2007]],
 
         [[ 0.1920],
          [ 0.2012],
          [ 0.2356],
          ...,
          [ 0.2230],
          [ 0.2293],
          [ 0.2321]],
 
         ...,
 
         [[-0.2197],
          [-0.2880],
          [-0.3161],
          ...,
          [-0.2657],
          [-0.2775],
          [-0.2369]],
 
         [[ 0.3287],
          [ 0.2476],
          [ 0.1606],
          ...,
          [ 0.1306],
          [ 0.1280],
          [ 0.1124]],
 
         [[-0.2532],
          [-0.3038],
          [-0.3364],
          ...,
          [-0.6732],
          [-0.6490],
          [-0.6315]]], grad_fn=<ViewBackward0>)}

### Pytorch generated structure of revnet

In [8]:
from torchview import draw_graph
model_graph = draw_graph(revnet, input_data=collated, device='meta')
model_graph.visual_graph

ValueError: maximum supported dimension for an ndarray is currently 64, found 365

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/logs

### Excluded basin notes

03281100: NaN runoff ratio

I have not included most of the basins because it says Year is somehow missing.
I gotta diagnose it later